# Session 2-B — LSTM 기반 터보팬 잔여 유효 수명(RUL) 예측 실습 노트북

## NASA C-MAPSS 데이터 기반 시퀀스 학습 → RUL 회귀 예측

**데이터**: `h2_data_copy/CMaps/` 내 FD001 (단일 운영조건, HPC degradation)  
**센서**: 21개 중 유효 14개 (상수 7개 제외)  
**방법**: Sliding window 시퀀스 → LSTM → RUL 예측, Early Stopping + Dropout 적용

> 참고 교재: *Deep Learning with Python 2nd Ed.* (François Chollet) | *ML for Time-Series with Python*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import os, warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. 데이터 로드 및 전처리

In [ ]:
DATA_DIR = 'h2_data_copy/CMaps/'
cols = ['unit', 'cycle'] + [f'op_{i}' for i in range(1,4)] + [f'sensor_{i}' for i in range(1,22)]

# FD001 로드
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_FD001.txt'), sep=' ', header=None)
train_df.dropna(axis=1, how='all', inplace=True)
train_df.columns = cols

test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_FD001.txt'), sep=' ', header=None)
test_df.dropna(axis=1, how='all', inplace=True)
test_df.columns = cols

rul_true = pd.read_csv(os.path.join(DATA_DIR, 'RUL_FD001.txt'), sep=' ', header=None)
rul_true.dropna(axis=1, how='all', inplace=True)
rul_true.columns = ['rul']

# RUL 라벨 생성 (train)
max_cycles = train_df.groupby('unit')['cycle'].max().reset_index()
max_cycles.columns = ['unit', 'max_cycle']
train_df = train_df.merge(max_cycles, on='unit')
train_df['rul'] = (train_df['max_cycle'] - train_df['cycle']).clip(upper=125)
train_df.drop('max_cycle', axis=1, inplace=True)

# 상수 센서 제거
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
constant_sensors = [s for s in sensor_cols if train_df[s].std() < 0.01]
useful_sensors = [s for s in sensor_cols if s not in constant_sensors]
print(f'Useful sensors ({len(useful_sensors)}): {useful_sensors}')

# 정규화
scaler = MinMaxScaler()
train_df[useful_sensors] = scaler.fit_transform(train_df[useful_sensors])
test_df[useful_sensors] = scaler.transform(test_df[useful_sensors])

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'RUL clipped distribution: mean={train_df.rul.mean():.1f}, std={train_df.rul.std():.1f}')

## 2. Sliding Window 시퀀스 생성

In [ ]:
SEQUENCE_LENGTH = 30

def create_sequences(df, sensors, seq_len):
    """각 엔진별로 sliding window 시퀀스 생성"""
    X, y, engine_ids = [], [], []
    for unit in df['unit'].unique():
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        values = unit_data[sensors].values
        targets = unit_data['rul'].values if 'rul' in unit_data.columns else None
        
        for i in range(seq_len, len(values) + 1):
            X.append(values[i-seq_len:i])
            if targets is not None:
                y.append(targets[i-1])
            engine_ids.append(unit)
    
    X = np.array(X)
    engine_ids = np.array(engine_ids)
    if targets is not None:
        y = np.array(y)
        return X, y, engine_ids
    return X, engine_ids

# Train 시퀀스
X_train, y_train, train_ids = create_sequences(train_df, useful_sensors, SEQUENCE_LENGTH)
print(f'Train sequences: {X_train.shape}, Targets: {y_train.shape}')

# Test: 각 엔진의 마지막 시퀀스 추출 (짧은 엔진은 패딩)
X_test = []
test_engine_ids = sorted(test_df['unit'].unique())

for eid in test_engine_ids:
    engine_data = test_df[test_df['unit'] == eid].sort_values('cycle')
    values = engine_data[useful_sensors].values
    
    if len(values) >= SEQUENCE_LENGTH:
        seq = values[-SEQUENCE_LENGTH:]
    else:
        # 패딩: 앞부분을 첫 번째 값으로 채움
        pad = np.tile(values[0], (SEQUENCE_LENGTH - len(values), 1))
        seq = np.vstack([pad, values])
    X_test.append(seq)

X_test = np.array(X_test)
y_test = rul_true['rul'].values

print(f'Test sequences (last per engine): {X_test.shape}')
print(f'Test RUL range: [{y_test.min()}, {y_test.max()}], mean: {y_test.mean():.1f}')

In [ ]:
# Train/Val split (엔진 단위)
all_engines = sorted(train_df['unit'].unique())
np.random.seed(42)
np.random.shuffle(all_engines)
n_val = int(len(all_engines) * 0.2)
val_engines = set(all_engines[:n_val])
train_engines = set(all_engines[n_val:])

train_mask = np.isin(train_ids, list(train_engines))
val_mask = np.isin(train_ids, list(val_engines))

X_tr = X_train[train_mask]
y_tr = y_train[train_mask]
X_va = X_train[val_mask]
y_va = y_train[val_mask]

print(f'Train: {X_tr.shape}, Val: {X_va.shape}')
print(f'Train RUL: mean={y_tr.mean():.1f}, std={y_tr.std():.1f}')
print(f'Val RUL: mean={y_va.mean():.1f}, std={y_va.std():.1f}')

## 3. LSTM 모델 정의

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

input_size = len(useful_sensors)
model = LSTMModel(input_size, hidden_size=64, num_layers=2, dropout=0.2).to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. 학습 (Early Stopping 적용)

In [ ]:
# DataLoader
train_dataset = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr))
val_dataset = TensorDataset(torch.FloatTensor(X_va), torch.FloatTensor(y_va))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 10
n_epochs = 100

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_batch)
    train_losses.append(epoch_loss / len(X_tr))
    
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            val_loss += criterion(pred, y_batch).item() * len(X_batch)
        val_losses.append(val_loss / len(X_va))
    
    scheduler.step(val_losses[-1])
    
    # Early Stopping
    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Train: {train_losses[-1]:.4f} | Val: {val_losses[-1]:.4f} | Best: {best_val_loss:.4f} | Patience: {patience_counter}/{PATIENCE}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

# Best model 로드
model.load_state_dict(best_state)
print(f'\nBest validation RMSE: {np.sqrt(best_val_loss):.4f}')

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='Train MSE', linewidth=1.5)
axes[0].plot(val_losses, label='Val MSE', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('LSTM Training Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(np.sqrt(train_losses), label='Train RMSE', linewidth=1.5)
axes[1].plot(np.sqrt(val_losses), label='Val RMSE', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE (cycles)')
axes[1].set_title('RMSE over Epochs')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('h2_fig_lstm_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Test 세트 평가

In [ ]:
# Test 예측
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_pred = model(X_test_tensor).cpu().numpy()

y_pred = np.clip(y_pred, 0, 125)  # RUL 범위 클리핑

# 메트릭
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f'=== LSTM RUL Prediction Results ===')
print(f'RMSE: {rmse:.2f} cycles')
print(f'MAE:  {mae:.2f} cycles')
print(f'\nTrue RUL  - mean: {y_test.mean():.1f}, std: {y_test.std():.1f}')
print(f'Pred RUL  - mean: {y_pred.mean():.1f}, std: {y_pred.std():.1f}')

In [ ]:
# NASA Scoring Function
def nasa_score(true, pred):
    """NASA C-MAPSS 비대칭 점수 함수
    늦은 예측(pred > true)에 더 큰 패널티"""
    d = pred - true
    score = np.where(d < 0,
                     np.exp(-d/13) - 1,   # 이른 예측: 작은 패널티
                     np.exp(d/10) - 1)     # 늦은 예측: 큰 패널티
    return np.sum(score)

score = nasa_score(y_test, y_pred)
print(f'NASA Score: {score:.1f}')
print(f'\n[NASA Score 해석]')
print(f'값이 작을수록 좋은 예측.')
print(f'비대칭 패널티: 늦은 예측(위험) > 이른 예측(보수적)')
print(f'  이른 예측(d<0): exp(-d/13)-1 → 완만한 증가')
print(f'  늦은 예측(d>0): exp(d/10)-1  → 급격한 증가')

In [ ]:
# 예측 vs 실제 산점도
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue')
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('True RUL')
axes[0].set_ylabel('Predicted RUL')
axes[0].set_title(f'RUL Prediction (RMSE={rmse:.2f}, MAE={mae:.2f})')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 오차 분포
errors = y_pred - y_test
axes[1].hist(errors, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Prediction Error (Pred - True)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Error Distribution (mean={errors.mean():.2f})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('h2_fig_lstm_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n[관찰] RMSE {rmse:.2f} cycles는 FD001에서 양호한 수준')
print(f'[관찰] 오차 분포가 0 근처에 집중되어 있고, 약간 보수적(이른 예측) 경향')

## 6. 엔진별 RUL 예측 궤적

In [ ]:
# 특정 엔진의 시간별 RUL 예측
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sample_engines = [1, 10, 20, 35, 50, 69]

model.eval()
for idx, eid in enumerate(sample_engines):
    ax = axes[idx // 3][idx % 3]
    engine_data = train_df[train_df.unit == eid].sort_values('cycle')
    values = engine_data[useful_sensors].values
    true_rul = engine_data['rul'].values
    
    # 모든 시점에서 예측
    preds = []
    for i in range(SEQUENCE_LENGTH, len(values) + 1):
        seq = torch.FloatTensor(values[i-SEQUENCE_LENGTH:i]).unsqueeze(0).to(device)
        with torch.no_grad():
            p = model(seq).cpu().item()
        preds.append(np.clip(p, 0, 125))
    
    cycles = engine_data['cycle'].values[SEQUENCE_LENGTH-1:]
    ax.plot(cycles, true_rul[SEQUENCE_LENGTH-1:], label='True RUL', linewidth=2, color='steelblue')
    ax.plot(cycles, preds, label='Predicted RUL', linewidth=1.5, color='coral', linestyle='--')
    ax.set_title(f'Engine #{eid}')
    ax.set_xlabel('Cycle')
    ax.set_ylabel('RUL')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('LSTM RUL Prediction Trajectories', fontsize=14)
plt.tight_layout()
plt.savefig('h2_fig_lstm_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n[관찰] 예측 RUL이 실제 RUL을 대체로 잘 따라감")
print("[관찰] 노화가 진행될수록(RUL이 낮아질수록) 예측이 더 정확해지는 경향")

## 7. Dropout 및 Early Stopping 효과 분석

In [ ]:
# Dropout 비교 실험
dropout_results = {}

for dropout_rate in [0.0, 0.1, 0.2, 0.3, 0.5]:
    model_exp = LSTMModel(input_size, hidden_size=64, num_layers=2, dropout=dropout_rate).to(device)
    opt = torch.optim.Adam(model_exp.parameters(), lr=1e-3)
    
    for epoch in range(30):  # 짧은 학습
        model_exp.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model_exp(X_batch)
            loss = criterion(pred, y_batch)
            opt.zero_grad()
            loss.backward()
            opt.step()
    
    # 평가
    model_exp.eval()
    with torch.no_grad():
        val_preds = model_exp(torch.FloatTensor(X_va).to(device)).cpu().numpy()
        test_preds = model_exp(torch.FloatTensor(X_test).to(device)).cpu().numpy()
    
    val_rmse = np.sqrt(mean_squared_error(y_va, val_preds))
    test_rmse = np.sqrt(mean_squared_error(y_test, np.clip(test_preds, 0, 125)))
    dropout_results[dropout_rate] = {'val_rmse': val_rmse, 'test_rmse': test_rmse}
    print(f'Dropout={dropout_rate:.1f} | Val RMSE: {val_rmse:.2f} | Test RMSE: {test_rmse:.2f}')

print(f'\n[분석] Dropout이 너무 높으면(0.5) 학습이 불충분해짐')
print(f'[분석] Dropout 0.1~0.3 범위에서 일반화 성능이 가장 좋음')

## 8. 요약

### LSTM RUL 예측 파이프라인
1. **전처리**: 14개 유효 센서, MinMaxScaler, RUL 클리핑(125)
2. **시퀀스**: 30-cycle sliding window → (batch, 30, 14) 형태
3. **모델**: 2-layer LSTM (hidden=64) + FC head
4. **정규화**: Dropout(0.2) + Early Stopping(patience=10)
5. **평가**: RMSE, MAE, NASA Score

### 핵심 인사이트
- **Sliding window**: 시퀀스 길이 30이 RUL 예측에 적합 (너무 길면 노화 전 데이터 포함)
- **RUL 클리핑(125)**: 초기 건강 구간의 노이즈를 제거해 학습 안정화
- **Early Stopping**: 검증 손실이 더 이상 개선되지 않으면 학습 중단, 과적합 방지
- **Dropout**: 은닉층 노드를 확률적으로 꺼서 모델이 특정 패턴에 과의존하지 않도록 규제
- **NASA Score**: 늦은 예측에 더 큰 패널티를 부여하는 비대칭 메트릭

### 교재 연결
- *Deep Learning with Python* Ch.5: Dropout은 은행 사기 방지에서 영감받은 정규화 기법
- *Deep Learning with Python* Ch.10: 시계열 예측을 위한 LSTM 아키텍처
- *ML for Time-Series*: 시계열 ML 플라이휠(데이터→특징→모델→평가→개선)의 실전 적용